# Solutions: Notebook 06 — Advanced Diagnostics for Quantile Regression

Complete solutions for all exercises in Notebook 06.

In [ ]:
# Standard libraries
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Statistical libraries
from statsmodels.nonparametric.smoothers_lowess import lowess

from panelbox.diagnostics.quantile import (
    AdvancedDiagnostics,
)

# PanelBox imports
from panelbox.models.quantile import PooledQuantile

# Visualization configuration
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 4)

# Reproducibility
np.random.seed(42)

# Define paths (relative to solutions/)
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOTS_DIR = OUTPUT_DIR / "plots"
RESULTS_DIR = OUTPUT_DIR / "results"
REPORTS_DIR = OUTPUT_DIR / "reports"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Load data and prepare baseline models (same as Notebook 06)
data = pd.read_csv(DATA_DIR / "card_education.csv")

y = data["lwage"].values
var_names = ["const", "educ", "exper", "exper_sq"]
X = np.column_stack(
    [np.ones(len(data)), data["educ"].values, data["exper"].values, data["exper"].values ** 2]
)
entity_id = data["id"].values

# Fit models at multiple quantiles
tau_list = [0.1, 0.25, 0.5, 0.75, 0.9]
qr_results = {}
for tau in tau_list:
    model = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
    qr_results[tau] = model.fit(se_type="cluster")

# Augmented model
var_names_aug = ["const", "educ", "exper", "exper_sq", "female", "married", "educ_x_exper"]
X_aug = np.column_stack(
    [
        np.ones(len(data)),
        data["educ"].values,
        data["exper"].values,
        data["exper"].values ** 2,
        data["female"].values,
        data["married"].values,
        data["educ"].values * data["exper"].values,
    ]
)

qr_results_aug = {}
for tau in tau_list:
    model = PooledQuantile(y, X_aug, entity_id=entity_id, quantiles=tau)
    qr_results_aug[tau] = model.fit(se_type="cluster")


# ResultWrapper helper class
class SimpleResult:
    def __init__(self, params, converged=True):
        self.params = params
        self.converged = converged


class SimpleModel:
    def __init__(self, X, y):
        self.X = X
        self.y = y


class ResultWrapper:
    def __init__(self, qr_results_dict, X, y):
        self.results = {}
        for tau, res in qr_results_dict.items():
            self.results[tau] = SimpleResult(params=res.params.ravel(), converged=res.converged)
        self.model = SimpleModel(X, y)


# Create wrappers
result_wrapped = ResultWrapper(qr_results, X, y)
result_aug_wrapped = ResultWrapper(qr_results_aug, X_aug, y)


# Helper: check loss
def check_loss(u, tau):
    return np.sum(u * (tau - (u < 0).astype(float)))


def pseudo_r_squared(y, X, params, tau):
    resid_full = y - X @ params
    loss_full = check_loss(resid_full, tau)
    q_tau = np.quantile(y, tau)
    resid_restricted = y - q_tau
    loss_restricted = check_loss(resid_restricted, tau)
    return 1 - loss_full / loss_restricted if loss_restricted != 0 else 0


print(f"Data loaded: {data.shape}")
print(f"Basic model: {X.shape[1]} variables")
print(f"Augmented model: {X_aug.shape[1]} variables")
print(f"Quantiles estimated: {tau_list}")

---

## Exercise 1: Run Full Diagnostics on Augmented Model (Easy)

**Task**: Using the augmented model (with `female`, `married`, and `educ*exper` interaction), run the full diagnostic suite at $\tau=0.5$. Interpret each test result in plain language.

In [ ]:
# Exercise 1 Solution: Full diagnostics on augmented model

# Step 1: Create ResultWrapper for augmented model (already done in setup)
print("FULL DIAGNOSTICS ON AUGMENTED MODEL")
print("Model: lwage ~ const + educ + exper + exper_sq + female + married + educ*exper")
print("=" * 70)

# Step 2: Run AdvancedDiagnostics
diag_aug = AdvancedDiagnostics(result_aug_wrapped, verbose=False)
report_aug = diag_aug.run_all_diagnostics(tau=0.5)

# Step 3: Print health score
print(f"\nHealth Score: {report_aug.health_score:.1%}")
print(f"Health Status: {report_aug.health_status.upper()}")

if report_aug.health_score >= 0.8:
    print("Interpretation: Model is HEALTHY - diagnostics look good.")
elif report_aug.health_score >= 0.5:
    print("Interpretation: Model has WARNINGS - some diagnostics need attention.")
else:
    print("Interpretation: Model has SERIOUS ISSUES - consider re-specification.")

# Step 4: Interpret each test
print(f"\n{'Test':<35} {'Status':<10} {'Statistic':<15} {'P-value':<10}")
print("-" * 70)

for d in report_aug.diagnostics:
    stat_str = f"{d.statistic:.4f}" if not np.isnan(d.statistic) else "N/A"
    pval_str = f"{d.p_value:.4f}" if not np.isnan(d.p_value) else "N/A"
    print(f"{d.test_name:<35} {d.status.upper():<10} {stat_str:<15} {pval_str:<10}")

# Plain-language interpretation for each test
print("\n" + "=" * 70)
print("PLAIN-LANGUAGE INTERPRETATION")
print("=" * 70)

interpretations = {
    "Khmaladze Specification Test": {
        "pass": "The model functional form appears correct. No evidence of missing variables or wrong specification.",
        "warning": "Marginal evidence of misspecification. Consider adding interaction terms or polynomial terms.",
        "fail": "The model is misspecified. Important variables may be missing, or the functional form is wrong. Try adding more controls, interactions, or nonlinear terms.",
    },
    "He-Zhu Heteroscedasticity Test": {
        "pass": "No significant heteroskedasticity detected. Conditional variance appears constant across X.",
        "warning": "Some evidence of heteroskedasticity. Standard errors may be affected; consider robust SEs.",
        "fail": "Strong heteroskedasticity detected. The variance of the outcome depends on covariates. Consider a location-scale model (Notebook 05) for more efficient estimation.",
    },
    "Outlier Detection": {
        "pass": "Few outliers detected. The data is well-behaved at this quantile.",
        "warning": "Moderate number of outliers. Some observations have unusual residuals; investigate for data quality.",
        "fail": "Many outliers detected. This may indicate data quality issues, or that the model does not adequately describe these observations. Consider winsorizing or investigating flagged observations.",
    },
    "Influence Diagnostics": {
        "pass": "No observation disproportionately influences the results. Estimates are stable.",
        "warning": "Some observations have moderate influence. Results may change slightly if these are removed.",
        "fail": "Several observations strongly influence the coefficients. Run a sensitivity analysis by excluding the most influential observations to check robustness.",
    },
    "Convergence Check": {
        "pass": "The optimization converged successfully. Estimates are numerically reliable.",
        "warning": "Convergence is marginal. Consider using tighter tolerances or different starting values.",
        "fail": "Convergence issues detected. The estimates may not be reliable. Try scaling covariates, changing the optimization method, or relaxing the tolerance.",
    },
    "Monotonicity Check": {
        "pass": "No quantile crossing detected. Predictions are monotone across quantiles.",
        "warning": "Minor quantile crossing observed. May be acceptable but consider rearrangement.",
        "fail": "Significant quantile crossing. Predicted quantiles are not monotone. Use monotonicity correction (rearrangement, constrained QR) or a location-scale model.",
    },
}

for d in report_aug.diagnostics:
    interp = interpretations.get(d.test_name, {}).get(d.status, d.message)
    print(f"\n[{d.status.upper():>4}] {d.test_name}")
    print(f"       {interp}")

---

## Exercise 2: Custom Outlier Detection with IQR Method (Easy)

**Task**: Implement an outlier detection method using the IQR (interquartile range) of residuals. Compare with the MAD method.

In [ ]:
# Exercise 2 Solution: IQR-based outlier detection
print("OUTLIER DETECTION: IQR vs MAD")
print("=" * 60)

# Compute residuals for tau=0.5
params_50 = qr_results[0.5].params.ravel()
residuals_50 = y - X @ params_50

# --- IQR Method ---
Q1 = np.percentile(residuals_50, 25)
Q3 = np.percentile(residuals_50, 75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR
outliers_iqr = (residuals_50 < lower_fence) | (residuals_50 > upper_fence)

print("IQR Method:")
print(f"  Q1 = {Q1:.4f}")
print(f"  Q3 = {Q3:.4f}")
print(f"  IQR = {IQR:.4f}")
print(f"  Lower fence = {lower_fence:.4f}")
print(f"  Upper fence = {upper_fence:.4f}")
print(f"  Outliers: {np.sum(outliers_iqr)} ({100 * np.mean(outliers_iqr):.2f}%)")

# --- MAD Method ---
mad = np.median(np.abs(residuals_50 - np.median(residuals_50)))
std_resid_mad = residuals_50 / (1.4826 * mad)
threshold_mad = 3.0
outliers_mad = np.abs(std_resid_mad) > threshold_mad

print(f"\nMAD Method (threshold={threshold_mad}):")
print(f"  MAD = {mad:.4f}")
print(f"  Adjusted MAD (1.4826*MAD) = {1.4826 * mad:.4f}")
print(f"  Outliers: {np.sum(outliers_mad)} ({100 * np.mean(outliers_mad):.2f}%)")

# --- Comparison ---
both = outliers_iqr & outliers_mad
only_iqr = outliers_iqr & ~outliers_mad
only_mad = outliers_mad & ~outliers_iqr

print("\nComparison:")
print(f"  Both methods flag:    {np.sum(both)} observations")
print(f"  Only IQR flags:       {np.sum(only_iqr)} observations")
print(f"  Only MAD flags:       {np.sum(only_mad)} observations")
print(f"  Jaccard similarity:   {np.sum(both) / np.sum(outliers_iqr | outliers_mad):.3f}")

In [ ]:
# Visualization: compare the two methods
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Residual distribution with fences
ax = axes[0]
ax.hist(residuals_50, bins=60, density=True, alpha=0.6, color="steelblue", edgecolor="white")
ax.axvline(lower_fence, color="red", linestyle="--", linewidth=2, label="IQR fences")
ax.axvline(upper_fence, color="red", linestyle="--", linewidth=2)
mad_lower = -threshold_mad * 1.4826 * mad
mad_upper = threshold_mad * 1.4826 * mad
ax.axvline(mad_lower, color="orange", linestyle=":", linewidth=2, label="MAD fences")
ax.axvline(mad_upper, color="orange", linestyle=":", linewidth=2)
ax.set_xlabel("Residuals", fontsize=12)
ax.set_ylabel("Density", fontsize=12)
ax.set_title("Residual Distribution with Outlier Fences", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Plot 2: Scatter of residuals, colored by method
ax = axes[1]
colors = np.where(
    both, "red", np.where(only_iqr, "orange", np.where(only_mad, "purple", "steelblue"))
)
ax.scatter(range(len(residuals_50)), residuals_50, c=colors, alpha=0.4, s=10)
ax.axhline(0, color="black", linewidth=0.5)
# Add legend manually
from matplotlib.lines import Line2D

legend_elements = [
    Line2D(
        [0], [0], marker="o", color="w", markerfacecolor="steelblue", markersize=8, label="Normal"
    ),
    Line2D(
        [0], [0], marker="o", color="w", markerfacecolor="red", markersize=8, label="Both methods"
    ),
    Line2D(
        [0], [0], marker="o", color="w", markerfacecolor="orange", markersize=8, label="IQR only"
    ),
    Line2D(
        [0], [0], marker="o", color="w", markerfacecolor="purple", markersize=8, label="MAD only"
    ),
]
ax.legend(handles=legend_elements, fontsize=9)
ax.set_xlabel("Observation Index", fontsize=12)
ax.set_ylabel("Residual", fontsize=12)
ax.set_title("Outliers by Detection Method", fontsize=13, fontweight="bold")
ax.grid(alpha=0.3)

# Plot 3: Venn-style bar chart
ax = axes[2]
categories = ["Both", "IQR only", "MAD only", "Neither"]
counts = [np.sum(both), np.sum(only_iqr), np.sum(only_mad), np.sum(~outliers_iqr & ~outliers_mad)]
bar_colors = ["red", "orange", "purple", "steelblue"]
bars = ax.bar(categories, counts, color=bar_colors, edgecolor="white", linewidth=1.5)
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        str(count),
        ha="center",
        fontsize=11,
        fontweight="bold",
    )
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Outlier Classification Summary", fontsize=13, fontweight="bold")
ax.grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print("\nConclusion:")
print("- MAD is more robust to the outliers themselves, so its estimate of spread")
print("  is less inflated by extreme values.")
print("- IQR uses a fixed multiplier (1.5) on the interquartile range.")
print("- Both methods are robust alternatives to SD-based detection.")
print("- MAD typically flags more outliers in heavy-tailed distributions.")

---

## Exercise 3: Custom Weighted Health Score (Medium)

**Task**: Create a custom health score function with different weights for each test.

In [ ]:
# Exercise 3 Solution: Custom weighted health score
print("CUSTOM WEIGHTED HEALTH SCORE")
print("=" * 60)


def weighted_health_score(diagnostics, weights=None):
    """
    Compute a weighted health score from diagnostic results.

    Parameters
    ----------
    diagnostics : list of DiagnosticResult
        Results from AdvancedDiagnostics.run_all_diagnostics()
    weights : dict, optional
        Mapping of test_name -> weight. Tests not in dict get weight=1.

    Returns
    -------
    dict with 'score', 'status', 'details'
    """
    if weights is None:
        weights = {}  # all equal

    status_scores = {"pass": 1.0, "warning": 0.5, "fail": 0.0}

    total_weight = 0
    weighted_sum = 0
    details = []

    for d in diagnostics:
        w = weights.get(d.test_name, 1.0)
        s = status_scores.get(d.status, 0.0)
        weighted_sum += w * s
        total_weight += w
        details.append(
            {"test": d.test_name, "status": d.status, "weight": w, "contribution": w * s}
        )

    score = weighted_sum / total_weight if total_weight > 0 else 0

    if score >= 0.8:
        status = "GOOD"
    elif score >= 0.5:
        status = "FAIR"
    else:
        status = "POOR"

    return {"score": score, "status": status, "total_weight": total_weight, "details": details}


# Custom weights: specification and convergence are most important
custom_weights = {
    "Khmaladze Specification Test": 3.0,
    "He-Zhu Heteroscedasticity Test": 2.0,
    "Outlier Detection": 1.0,
    "Influence Diagnostics": 1.0,
    "Convergence Check": 3.0,
    "Monotonicity Check": 2.0,
}

# Run diagnostics
diag_ex3 = AdvancedDiagnostics(result_wrapped, verbose=False)
report_ex3 = diag_ex3.run_all_diagnostics(tau=0.5)

# Default (equal weights)
result_default = weighted_health_score(report_ex3.diagnostics)

# Custom weights
result_custom = weighted_health_score(report_ex3.diagnostics, weights=custom_weights)

# Print comparison
print(f"\n{'':<35} {'Default (equal)':<18} {'Custom (weighted)':<18}")
print("-" * 71)
print(f"{'Health Score':<35} {result_default['score']:.1%}{'':<13} {result_custom['score']:.1%}")
print(f"{'Health Status':<35} {result_default['status']:<18} {result_custom['status']:<18}")

print("\nDetailed Breakdown (Custom Weights):")
print(f"{'Test':<35} {'Status':<10} {'Weight':<10} {'Contribution':<12}")
print("-" * 67)
for detail in result_custom["details"]:
    print(
        f"{detail['test']:<35} {detail['status'].upper():<10} {detail['weight']:<10.1f} {detail['contribution']:<12.2f}"
    )
print(
    f"{'TOTAL':<35} {'':<10} {result_custom['total_weight']:<10.1f} {sum(d['contribution'] for d in result_custom['details']):<12.2f}"
)

print(
    f"\nWeighted Score = {sum(d['contribution'] for d in result_custom['details']):.2f} / {result_custom['total_weight']:.1f} = {result_custom['score']:.1%}"
)

print("\nKey Insight: Custom weights let you emphasize the diagnostics most")
print("relevant to your research question. If correct specification is critical,")
print("give the Khmaladze test a high weight.")

---

## Exercise 4: Detect Misspecification via Simulation (Medium)

**Task**: Simulate data with a known quadratic DGP, fit a linear-only model, and show that the Khmaladze test detects the misspecification. Then add the quadratic term and re-test.

In [ ]:
# Exercise 4 Solution: Simulate misspecified model
print("MISSPECIFICATION DETECTION VIA SIMULATION")
print("=" * 60)

np.random.seed(42)

# True DGP: Y = 1 + 2*X + 3*X^2 + epsilon
n_sim = 2000
X_raw = np.random.randn(n_sim)
epsilon = np.random.randn(n_sim) * (1 + 0.5 * np.abs(X_raw))  # heteroskedastic errors
y_sim = 1.0 + 2.0 * X_raw + 3.0 * X_raw**2 + epsilon

# Create entity IDs for clustering (just sequential groups)
entity_sim = np.repeat(np.arange(200), 10)

print("True DGP: Y = 1.0 + 2.0*X + 3.0*X^2 + epsilon")
print(f"N = {n_sim}")

# --- Model 1: Linear-only (MISSPECIFIED) ---
print(f"\n{'=' * 60}")
print("Model 1: LINEAR ONLY (Misspecified - omitting X^2)")
print(f"{'=' * 60}")

X_linear = np.column_stack([np.ones(n_sim), X_raw])

# Fit at multiple quantiles
tau_sim = [0.1, 0.25, 0.5, 0.75, 0.9]
qr_linear = {}
for tau in tau_sim:
    model = PooledQuantile(y_sim, X_linear, entity_id=entity_sim, quantiles=tau)
    qr_linear[tau] = model.fit(se_type="cluster")

# Wrap and run Khmaladze test
wrapped_linear = ResultWrapper(qr_linear, X_linear, y_sim)
diag_linear = AdvancedDiagnostics(wrapped_linear, verbose=False)
diag_linear.test_specification(tau=0.5)

khm_linear = diag_linear.diagnostics[-1]
print(f"Khmaladze statistic: {khm_linear.statistic:.4f}")
print(f"P-value:             {khm_linear.p_value:.4f}")
print(f"Status:              {khm_linear.status.upper()}")
print(f"Message:             {khm_linear.message}")

if khm_linear.status == "fail":
    print("\n--> Khmaladze CORRECTLY detects the misspecification!")
else:
    print("\n--> Khmaladze did not reject (Type II error or insufficient power).")

# --- Model 2: Correct model (with X^2) ---
print(f"\n{'=' * 60}")
print("Model 2: CORRECT MODEL (including X^2)")
print(f"{'=' * 60}")

X_correct = np.column_stack([np.ones(n_sim), X_raw, X_raw**2])

qr_correct = {}
for tau in tau_sim:
    model = PooledQuantile(y_sim, X_correct, entity_id=entity_sim, quantiles=tau)
    qr_correct[tau] = model.fit(se_type="cluster")

wrapped_correct = ResultWrapper(qr_correct, X_correct, y_sim)
diag_correct = AdvancedDiagnostics(wrapped_correct, verbose=False)
diag_correct.test_specification(tau=0.5)

khm_correct = diag_correct.diagnostics[-1]
print(f"Khmaladze statistic: {khm_correct.statistic:.4f}")
print(f"P-value:             {khm_correct.p_value:.4f}")
print(f"Status:              {khm_correct.status.upper()}")
print(f"Message:             {khm_correct.message}")

if khm_correct.status == "pass":
    print("\n--> Khmaladze correctly does NOT reject for the correct model!")
else:
    print("\n--> Khmaladze still rejects (may be due to heteroskedastic errors).")

# Compare Pseudo-R2
pr2_linear = pseudo_r_squared(y_sim, X_linear, qr_linear[0.5].params.ravel(), 0.5)
pr2_correct = pseudo_r_squared(y_sim, X_correct, qr_correct[0.5].params.ravel(), 0.5)

print("\nPseudo-R2 Comparison:")
print(f"  Linear-only: {pr2_linear:.4f}")
print(f"  With X^2:    {pr2_correct:.4f}")
print(f"  Improvement: {pr2_correct - pr2_linear:.4f}")

In [ ]:
# Visualization: misspecification is visible in residual plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear model residuals
ax = axes[0]
params_lin = qr_linear[0.5].params.ravel()
fitted_lin = X_linear @ params_lin
resid_lin = y_sim - fitted_lin
ax.scatter(X_raw, resid_lin, alpha=0.3, s=10, color="steelblue")
ax.axhline(0, color="red", linestyle="--", linewidth=2)
try:
    smooth_lin = lowess(resid_lin, X_raw, frac=0.3)
    ax.plot(smooth_lin[:, 0], smooth_lin[:, 1], "orange", linewidth=3, label="Lowess")
    ax.legend(fontsize=11)
except:
    pass
ax.set_xlabel("X", fontsize=12)
ax.set_ylabel("Residuals", fontsize=12)
ax.set_title(
    "Linear Only (Misspecified)\nClear quadratic pattern in residuals",
    fontsize=13,
    fontweight="bold",
)
ax.grid(alpha=0.3)

# Correct model residuals
ax = axes[1]
params_corr = qr_correct[0.5].params.ravel()
fitted_corr = X_correct @ params_corr
resid_corr = y_sim - fitted_corr
ax.scatter(X_raw, resid_corr, alpha=0.3, s=10, color="steelblue")
ax.axhline(0, color="red", linestyle="--", linewidth=2)
try:
    smooth_corr = lowess(resid_corr, X_raw, frac=0.3)
    ax.plot(smooth_corr[:, 0], smooth_corr[:, 1], "orange", linewidth=3, label="Lowess")
    ax.legend(fontsize=11)
except:
    pass
ax.set_xlabel("X", fontsize=12)
ax.set_ylabel("Residuals", fontsize=12)
ax.set_title(
    "Correct Model (with X$^2$)\nNo systematic pattern in residuals", fontsize=13, fontweight="bold"
)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("The left plot shows a clear quadratic pattern in the residuals,")
print("confirming the misspecification that Khmaladze detected.")
print("The right plot shows random scatter, as expected for a correct model.")

---

## Exercise 5: Compare Diagnostics Across Quantiles (Hard)

**Task**: Run the full diagnostic suite at $\tau \in \{0.1, 0.25, 0.5, 0.75, 0.9\}$. Create a summary table and a heatmap of diagnostic statuses.

In [ ]:
# Exercise 5 Solution: Diagnostics across multiple quantiles
print("DIAGNOSTICS ACROSS MULTIPLE QUANTILES")
print("=" * 70)

tau_grid = [0.1, 0.25, 0.5, 0.75, 0.9]
multi_reports = {}

for tau in tau_grid:
    diag_tau = AdvancedDiagnostics(result_wrapped, verbose=False)
    multi_reports[tau] = diag_tau.run_all_diagnostics(tau=tau)

# Build summary table as DataFrame
test_names = [d.test_name for d in multi_reports[0.5].diagnostics]

# Status table
status_data = {}
for tau in tau_grid:
    status_data[f"tau={tau}"] = [d.status.upper() for d in multi_reports[tau].diagnostics]

status_df = pd.DataFrame(status_data, index=test_names)
print("\nDiagnostic Status by Quantile:")
display(status_df)

# Statistics table
stat_data = {}
for tau in tau_grid:
    stat_data[f"tau={tau}"] = [
        f"{d.statistic:.4f}" if not np.isnan(d.statistic) else "N/A"
        for d in multi_reports[tau].diagnostics
    ]

stat_df = pd.DataFrame(stat_data, index=test_names)
print("\nDiagnostic Statistics by Quantile:")
display(stat_df)

# Health scores
print("\nHealth Scores by Quantile:")
for tau in tau_grid:
    print(
        f"  tau={tau:.2f}: {multi_reports[tau].health_score:.1%} ({multi_reports[tau].health_status.upper()})"
    )

# Check for quantile-specific issues
print("\nQuantile-Specific Analysis:")
for test_idx, test_name in enumerate(test_names):
    statuses = [multi_reports[tau].diagnostics[test_idx].status for tau in tau_grid]
    if len(set(statuses)) > 1:
        print(f"  {test_name}: VARIES across quantiles")
        for tau, status in zip(tau_grid, statuses):
            print(f"    tau={tau:.2f}: {status.upper()}")
    else:
        print(f"  {test_name}: Consistent ({statuses[0].upper()}) across all quantiles")

In [ ]:
# Heatmap visualization
status_map = {"PASS": 2, "WARNING": 1, "FAIL": 0}
heatmap_data = np.array(
    [
        [status_map.get(multi_reports[tau].diagnostics[i].status.upper(), -1) for tau in tau_grid]
        for i in range(len(test_names))
    ]
)

fig, ax = plt.subplots(figsize=(10, 6))

# Custom colormap: red (fail) -> yellow (warning) -> green (pass)
from matplotlib.colors import ListedColormap

cmap = ListedColormap(["#ff6b6b", "#ffd93d", "#6bcb77"])

im = ax.imshow(heatmap_data, cmap=cmap, vmin=0, vmax=2, aspect="auto")

# Labels
short_names = [
    name.replace(" Test", "")
    .replace("Diagnostics", "Diag.")
    .replace("Check", "Chk.")
    .replace("Detection", "Det.")
    for name in test_names
]
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names, fontsize=11)
ax.set_xticks(range(len(tau_grid)))
ax.set_xticklabels([f"tau={tau}" for tau in tau_grid], fontsize=11)

# Add text annotations
status_labels = {0: "FAIL", 1: "WARN", 2: "PASS"}
for i in range(len(test_names)):
    for j in range(len(tau_grid)):
        text_color = "white" if heatmap_data[i, j] == 0 else "black"
        ax.text(
            j,
            i,
            status_labels[heatmap_data[i, j]],
            ha="center",
            va="center",
            fontsize=10,
            fontweight="bold",
            color=text_color,
        )

ax.set_title("Diagnostic Status Heatmap Across Quantiles", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Quantile", fontsize=12)

# Add health score bar below
health_scores = [multi_reports[tau].health_score for tau in tau_grid]

plt.tight_layout()
plt.show()

# Additional plot: health score across quantiles
fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = [
    "#6bcb77" if s >= 0.8 else "#ffd93d" if s >= 0.5 else "#ff6b6b" for s in health_scores
]
bars = ax.bar(
    range(len(tau_grid)),
    [s * 100 for s in health_scores],
    color=colors_bar,
    edgecolor="white",
    linewidth=2,
)
ax.set_xticks(range(len(tau_grid)))
ax.set_xticklabels([f"tau={tau}" for tau in tau_grid], fontsize=11)
ax.set_ylabel("Health Score (%)", fontsize=12)
ax.set_title("Model Health Score Across Quantiles", fontsize=14, fontweight="bold")
ax.axhline(80, color="green", linestyle="--", alpha=0.5, label="Good threshold")
ax.axhline(50, color="orange", linestyle="--", alpha=0.5, label="Fair threshold")
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3, axis="y")

for bar, score in zip(bars, health_scores):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{score:.0%}",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

plt.tight_layout()
plt.show()

print("\nConclusion:")
print("- If problems are consistent across all quantiles, the issue is likely")
print("  fundamental (e.g., missing important variables).")
print("- If problems are quantile-specific (e.g., only at tails), the model")
print("  may be adequate for central quantiles but not for extremes.")

---

## Exercise 6: Full Estimate-Diagnose-Fix-Rediagnose Workflow (Hard)

**Task**: Implement a complete diagnostic workflow that automates the estimate-diagnose-fix-rediagnose cycle.

In [ ]:
# Exercise 6 Solution: Full diagnostic workflow
print("FULL ESTIMATE-DIAGNOSE-FIX-REDIAGNOSE WORKFLOW")
print("=" * 70)


def estimate_and_diagnose(y, X, entity_id, tau_list, label="Model"):
    """
    Estimate QR at multiple quantiles and run full diagnostics.

    Parameters
    ----------
    y : array, dependent variable
    X : array, design matrix
    entity_id : array, entity identifiers
    tau_list : list, quantile levels
    label : str, model label for display

    Returns
    -------
    dict with 'results', 'reports', 'summary'
    """
    print(f"\n--- {label} ---")
    print(f"    Variables: {X.shape[1]}, Observations: {X.shape[0]}")

    # Estimate
    qr_results = {}
    for tau in tau_list:
        model = PooledQuantile(y, X, entity_id=entity_id, quantiles=tau)
        qr_results[tau] = model.fit(se_type="cluster")

    # Wrap for diagnostics
    wrapped = ResultWrapper(qr_results, X, y)

    # Diagnose at each quantile
    reports = {}
    for tau in tau_list:
        diag = AdvancedDiagnostics(wrapped, verbose=False)
        reports[tau] = diag.run_all_diagnostics(tau=tau)

    # Build summary
    [d.test_name for d in reports[tau_list[0]].diagnostics]
    summary_rows = []
    for tau in tau_list:
        row = {"tau": tau, "health_score": reports[tau].health_score}
        for d in reports[tau].diagnostics:
            row[d.test_name] = d.status.upper()
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows).set_index("tau")

    # Pseudo-R2
    pr2_values = []
    for tau in tau_list:
        pr2 = pseudo_r_squared(y, X, qr_results[tau].params.ravel(), tau)
        pr2_values.append(pr2)
    summary_df["pseudo_r2"] = pr2_values

    avg_health = np.mean([reports[tau].health_score for tau in tau_list])
    print(f"    Average Health Score: {avg_health:.1%}")

    return {
        "results": qr_results,
        "reports": reports,
        "summary": summary_df,
        "avg_health": avg_health,
        "label": label,
    }


# Step 1: Simple model
print("STEP 1: INITIAL MODEL")
X_simple = np.column_stack(
    [
        np.ones(len(data)),
        data["educ"].values,
    ]
)
m1 = estimate_and_diagnose(y, X_simple, entity_id, tau_list, "Simple (educ only)")
print(m1["summary"])

In [ ]:
# Step 2: Add experience (based on recommendation: add variables)
print("STEP 2: ADD EXPERIENCE")
X_step2 = np.column_stack(
    [np.ones(len(data)), data["educ"].values, data["exper"].values, data["exper"].values ** 2]
)
m2 = estimate_and_diagnose(y, X_step2, entity_id, tau_list, "With experience")
print(m2["summary"])

In [ ]:
# Step 3: Add demographics + interactions (based on recommendation: add controls)
print("STEP 3: ADD DEMOGRAPHICS AND INTERACTIONS")
X_step3 = np.column_stack(
    [
        np.ones(len(data)),
        data["educ"].values,
        data["exper"].values,
        data["exper"].values ** 2,
        data["female"].values,
        data["married"].values,
        data["union"].values,
        data["educ"].values * data["exper"].values,
        data["educ"].values * data["female"].values,
    ]
)
m3 = estimate_and_diagnose(y, X_step3, entity_id, tau_list, "Full model + interactions")
print(m3["summary"])

In [ ]:
# Comparison table across all iterations
print("\n" + "=" * 70)
print("WORKFLOW COMPARISON")
print("=" * 70)

comparison_rows = []
for m in [m1, m2, m3]:
    row = {
        "Model": m["label"],
        "Variables": m["summary"].shape[1] - 1,  # exclude pseudo_r2 column
        "Avg Health": f"{m['avg_health']:.1%}",
        "Pseudo-R2 (med)": f"{m['summary']['pseudo_r2'].iloc[2]:.4f}",
    }
    # Count passes across all quantiles
    test_cols = [c for c in m["summary"].columns if c not in ("health_score", "pseudo_r2")]
    n_tests_total = len(test_cols) * len(tau_list)
    n_pass = sum((m["summary"][c] == "PASS").sum() for c in test_cols)
    row["Tests Passed"] = f"{n_pass}/{n_tests_total}"
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

# Plot health score evolution
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Average health score by model iteration
ax = axes[0]
models = [m1, m2, m3]
model_labels = [m["label"] for m in models]
health_scores_iter = [m["avg_health"] * 100 for m in models]
colors_iter = [
    "#ff6b6b" if h < 50 else "#ffd93d" if h < 80 else "#6bcb77" for h in health_scores_iter
]
bars = ax.bar(
    range(len(models)), health_scores_iter, color=colors_iter, edgecolor="white", linewidth=2
)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(
    ["Step 1\n(simple)", "Step 2\n(+ exper)", "Step 3\n(+ demographics)"], fontsize=10
)
ax.set_ylabel("Average Health Score (%)", fontsize=12)
ax.set_title("Model Health Score Evolution", fontsize=14, fontweight="bold")
ax.axhline(80, color="green", linestyle="--", alpha=0.4, label="Good")
ax.axhline(50, color="orange", linestyle="--", alpha=0.4, label="Fair")
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
ax.grid(alpha=0.3, axis="y")
for bar, score in zip(bars, health_scores_iter):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 2,
        f"{score:.0f}%",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

# Plot 2: Pseudo-R2 evolution across quantiles
ax = axes[1]
for m, ls_style, marker in zip(models, ["-", "--", "-."], ["o", "s", "D"]):
    pr2_vals = m["summary"]["pseudo_r2"].values
    ax.plot(
        tau_list, pr2_vals, ls_style, marker=marker, linewidth=2, markersize=7, label=m["label"]
    )
ax.set_xlabel("Quantile (tau)", fontsize=12)
ax.set_ylabel("Pseudo-R2", fontsize=12)
ax.set_title("Pseudo-R2 Across Quantiles by Model", fontsize=14, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Document findings
print("\nFINDINGS:")
print("-" * 60)
print("1. The simple model (educ only) has the lowest health score.")
print("   Khmaladze test rejects correct specification at all quantiles.")
print("2. Adding experience (with quadratic) improves Pseudo-R2 and may")
print("   improve some diagnostics, but specification issues persist.")
print("3. Adding demographics and interactions further improves the model.")
print("   Some tests may still fail due to the nature of the DGP.")
print("4. Persistent heteroskedasticity across all models suggests a")
print("   location-scale model would be more appropriate (Notebook 05).")

---

## Summary of Exercise Solutions

1. **Exercise 1**: The augmented model diagnostics show that adding controls improves pseudo-R2 but may not resolve all specification issues. Each test provides actionable guidance.

2. **Exercise 2**: IQR and MAD methods flag different outlier sets. MAD is more robust and typically flags more observations in heavy-tailed distributions. Both are valid alternatives to SD-based detection.

3. **Exercise 3**: Custom weighted health scores allow prioritizing diagnostics relevant to the research question. Specification (Khmaladze) and convergence should typically receive higher weights.

4. **Exercise 4**: The Khmaladze test successfully detects omitted quadratic terms in simulated data. Adding the correct term reduces the test statistic. Residual plots visually confirm the misspecification.

5. **Exercise 5**: Diagnostic heatmaps reveal whether problems are consistent or quantile-specific. Consistent failures suggest fundamental model issues; quantile-specific failures may be acceptable for certain analyses.

6. **Exercise 6**: The estimate-diagnose-fix-rediagnose workflow systematically improves model health. However, persistent heteroskedasticity suggests a location-scale model may be more appropriate than adding more controls to a standard QR.